# Practica 3. Machine Learning II
## TiendaAlimentos: preparacion de compras y patrones frecuentes

Mineria de Datos, Facultad de Ingenieria, UNAM.

Equipo: Lara Hernandez Angel Husiel, Lugo Manzano Rodrigo y Garcia Cortes Adolfo de Jesus.

Profesor: Mtro. Gerardo Gabriel Carrasco Zuñiga.

Para esta practica se usa el archivo de datos base que proporciono el profesor. Primero vamos a revisar las compras y preparar una matriz que indique que productos aparecen en cada cliente. Posteriormente se comparan Apriori y FP-Growth con el mismo soporte minimo.

Este avance comprende las secciones 1 y 2 de la guia. Las reglas de asociacion y las promociones corresponden a las partes posteriores.


# 1. Carga, exploracion e ingenieria de datos transaccionales
## 1.1. Preparacion y carga del archivo de datos base

Para esto se preparan las bibliotecas y despues se carga directamente el Excel. No hace falta comprimirlo. Al ejecutar la celda de carga se debe seleccionar el archivo de datos base que acompaña la guia.

El identificador del cliente se conserva para reconocer cada fila, pero no se utiliza como producto. Cabe mencionar que el archivo no tiene tickets ni fechas: cada registro representa la canasta registrada de un cliente en el periodo de la muestra, no necesariamente una sola compra individual.


In [14]:
%pip -q install mlxtend==0.23.4 openpyxl


In [15]:
import warnings
warnings.filterwarnings(
    'ignore',
    message=r'datetime[.]datetime[.]utcnow.*is deprecated.*',
    category=DeprecationWarning,
    module=r'jupyter_client[.]session'
)

from google.colab import files
from IPython.display import display
from io import BytesIO
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
from itertools import combinations
import platform
import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 15)
pd.set_option('display.precision', 4)
MIN_SUPPORT = 0.40
REPETICIONES = 30
print('Python:', platform.python_version())
for biblioteca in ['pandas', 'numpy', 'mlxtend', 'openpyxl']:
    print(biblioteca + ':', version(biblioteca))
print('Soporte minimo comun:', MIN_SUPPORT)


Python: 3.13.16
pandas: 2.2.3
numpy: 2.1.3
mlxtend: 0.23.4
openpyxl: 3.1.5
Soporte minimo comun: 0.4


In [16]:
archivos = files.upload()
excel = [nombre for nombre in archivos if nombre.lower().endswith('.xlsx')]
if len(excel) != 1:
    raise ValueError('Selecciona solamente el archivo Excel de datos base.')
df = pd.read_excel(BytesIO(archivos[excel[0]]), sheet_name='Hoja1', engine='openpyxl')
print('Archivo de datos base cargado.')
print('Filas y columnas:', df.shape)
display(df.head())


Saving TiendaAlimentos.xlsx to TiendaAlimentos (1).xlsx
Archivo de datos base cargado.
Filas y columnas: (30, 6)


,Cliente,ProductoA,ProductoB,ProductoC,ProductoD,ProductoE
0,Cliente 1,Tomates,Elotes,Pepino,Zanahoria,Pimientos
1,Cliente 2,Tomates,NaN,Pepino,Zanahoria,NaN
2,Cliente 3,Tomates,Elotes,Pepino,Zanahoria,Pimientos
3,Cliente 4,Tomates,Elotes,Pepino,NaN,NaN
4,Cliente 5,Tomates,Elotes,Pepino,Zanahoria,Pimientos


## 1.2. Inspeccion de los datos

Ahora bien, se revisan el numero de registros, las columnas, sus tipos y los valores vacios. Los encabezados ProductoA a ProductoE indican posiciones en el archivo; los nombres de los productos estan dentro de las celdas.

En esta base vamos a interpretar una celda vacia de producto como ausencia de compra. Esto coincide con el formato de las canastas proporcionadas y se deja como supuesto de preparacion, ya que no se incluye un diccionario adicional de datos. No se rellenan estos espacios con el producto mas frecuente, porque eso agregaria compras que no estan registradas.


In [17]:
if 'Cliente' not in df.columns or df.empty:
    raise ValueError('La base debe contener registros y la columna Cliente.')
columnas_producto = [columna for columna in df.columns if columna != 'Cliente']
n_transacciones = len(df)
inspeccion = pd.DataFrame({
    'Tipo': df.dtypes.astype(str),
    'Valores vacios': df.isna().sum(),
    'Proporcion de vacios': df.isna().mean(),
    'Porcentaje de vacios': df.isna().mean() * 100
}).rename_axis('Columna')
print('Registros:', n_transacciones)
print('Columnas:', df.columns.tolist())
print('Celdas vacias de toda la tabla:', int(df.isna().sum().sum()))
print('Porcentaje de celdas vacias de toda la tabla:', round(df.isna().to_numpy().mean() * 100, 4))
display(inspeccion)
clientes = df['Cliente'].astype('string').str.strip()
if clientes.isna().any() or clientes.eq('').any() or clientes.duplicated().any():
    raise ValueError('Revisa los identificadores: deben estar completos y ser unicos.')
print('Identificadores de cliente completos y unicos:', clientes.nunique())


Registros: 30
Columnas: ['Cliente', 'ProductoA', 'ProductoB', 'ProductoC', 'ProductoD', 'ProductoE']
Celdas vacias de toda la tabla: 32
Porcentaje de celdas vacias de toda la tabla: 17.7778


,Tipo,Valores vacios,Proporcion de vacios,Porcentaje de vacios
Columna,,,,
Cliente,object,0,0.0000,0.0000
ProductoA,object,9,0.3000,30.0000
ProductoB,object,7,0.2333,23.3333
ProductoC,object,8,0.2667,26.6667
ProductoD,object,4,0.1333,13.3333
ProductoE,object,4,0.1333,13.3333


Identificadores de cliente completos y unicos: 30


La base contiene 30 registros y seis columnas. Cliente no tiene valores vacios. En las columnas de productos hay 9, 7, 8, 4 y 4 celdas vacias, respectivamente: son 32 de 150 posiciones de producto, equivalentes a 21.33%. Si se considera toda la tabla, incluyendo Cliente, la proporcion es 32/180 = 17.78%. Se distinguen ambos calculos para que el denominador quede claro.

Dos clientes pueden tener la misma combinacion de productos. En ese caso se conservan las dos canastas, porque representan observaciones distintas y su frecuencia es parte de lo que se busca analizar.


## 1.3. Matriz binaria transaccional

Para cada cliente se toman los nombres de productos presentes y se retiran espacios al principio o al final. Despues se aplica TransactionEncoder. Cada columna corresponde a un producto y sus valores son True cuando aparece en la canasta y False cuando no aparece.

Si el mismo producto estuviera repetido dentro de una fila, se contaria una sola vez: aqui se estudia presencia de productos, no cantidades compradas. La matriz conserva una fila por cliente y no incluye al identificador como articulo.


In [18]:
transacciones = []
for fila in df[columnas_producto].itertuples(index=False, name=None):
    productos = sorted({str(valor).strip() for valor in fila if pd.notna(valor) and str(valor).strip()})
    transacciones.append(productos)

encoder = TransactionEncoder()
matriz = pd.DataFrame(
    encoder.fit(transacciones).transform(transacciones),
    columns=encoder.columns_,
    index=pd.Index(clientes.to_numpy(), name='Cliente'),
    dtype=bool
)
assert len(matriz) == n_transacciones
assert not matriz.isna().any().any()
assert all(tipo == np.dtype(bool) for tipo in matriz.dtypes)
assert matriz.sum(axis=1).tolist() == [len(canasta) for canasta in transacciones]
print('Matriz booleana, filas y productos:', matriz.shape)
print('Catalogo:', matriz.columns.tolist())
display(matriz)


Matriz booleana, filas y productos: (30, 5)
Catalogo: ['Elotes', 'Pepino', 'Pimientos', 'Tomates', 'Zanahoria']


,Elotes,Pepino,Pimientos,Tomates,Zanahoria
Cliente,,,,,
Cliente 1,True,True,True,True,True
Cliente 2,False,True,False,True,True
Cliente 3,True,True,True,True,True
Cliente 4,True,True,False,True,False
Cliente 5,True,True,True,True,True
Cliente 6,True,False,False,True,True
Cliente 7,True,True,True,True,True
Cliente 8,True,True,True,False,True
Cliente 9,True,False,True,False,True


## 1.4. Frecuencia de compra y soporte individual

El soporte individual es el numero de canastas que contienen un producto dividido entre el total de canastas. En este caso el denominador es 30. Por lo tanto, el soporte mide presencia en la muestra, no la cantidad de piezas vendidas ni el porcentaje de ingresos.


In [19]:
frecuencias = pd.DataFrame({
    'Canastas con el producto': matriz.sum().astype(int),
    'Soporte': matriz.mean(),
    'Soporte (%)': matriz.mean() * 100
}).rename_axis('Producto').reset_index()
frecuencias = frecuencias.sort_values(
    ['Canastas con el producto', 'Producto'], ascending=[False, True]
).reset_index(drop=True)
display(frecuencias)
max_frecuencia = frecuencias['Canastas con el producto'].max()
lideres = frecuencias.loc[frecuencias['Canastas con el producto'].eq(max_frecuencia), 'Producto'].tolist()
print('Productos con la mayor frecuencia:', ', '.join(lideres))
print('Canastas:', int(max_frecuencia), 'de', n_transacciones)
print('Soporte:', max_frecuencia / n_transacciones)


,Producto,Canastas con el producto,Soporte,Soporte (%)
0,Pimientos,26,0.8667,86.6667
1,Zanahoria,26,0.8667,86.6667
2,Elotes,23,0.7667,76.6667
3,Pepino,22,0.7333,73.3333
4,Tomates,21,0.7000,70.0000


Productos con la mayor frecuencia: Pimientos, Zanahoria
Canastas: 26 de 30
Soporte: 0.8666666666666667


## 1.5. Preguntas de la primera seccion

### ¿Cuantas transacciones y cuantos productos unicos componen el catalogo analizado?

Se tienen 30 canastas de clientes y cinco productos unicos: Elotes, Pepino, Pimientos, Tomates y Zanahoria. Para esta practica cada fila se utiliza como una transaccion. Como el archivo no tiene tickets ni fechas, lo que se analiza es la canasta registrada de cada cliente en la muestra disponible.

### ¿Cual es el producto con mayor frecuencia de compra individual y cual es su soporte empirico?

En este caso hay un empate: Pimientos y Zanahoria aparecen en 26 de las 30 canastas. Ambos tienen un soporte de 26/30 = 0.8667, aproximadamente 86.67%. Por lo tanto, no corresponde elegir solamente uno como el producto mas frecuente. Despues estan Elotes con 23, Pepino con 22 y Tomates con 21 canastas.


# 2. Extraccion de patrones frecuentes: Apriori vs. FP-Growth
## 2.1. Soporte minimo comun

Para esto se usa la misma matriz booleana con un soporte minimo de 0.40 en ambos algoritmos. Con 30 canastas, un conjunto debe aparecer en al menos 12 para considerarse frecuente. No se limita la longitud maxima de los conjuntos.

Aunque una instruccion de la guia permite usar uno u otro, se ejecutan los dos porque el objetivo y las preguntas piden compararlos. Las salidas se mantienen separadas para revisar sus coincidencias.


## 2.2. Patrones encontrados con Apriori

Apriori revisa primero los productos individuales y despues forma candidatos de mayor longitud. Se añade la longitud de cada conjunto y se conservan los nombres de los productos para que la tabla pueda interpretarse directamente.


In [20]:
patrones_apriori = apriori(matriz, min_support=MIN_SUPPORT, use_colnames=True)
patrones_apriori['longitud'] = patrones_apriori['itemsets'].apply(len)

def tabla_patrones(patrones):
    salida = patrones.copy()
    salida['Productos'] = salida['itemsets'].apply(lambda conjunto: ', '.join(sorted(conjunto)))
    salida['Canastas'] = np.rint(salida['support'] * n_transacciones).astype(int)
    salida['Soporte (%)'] = salida['support'] * 100
    return salida[['Productos', 'longitud', 'Canastas', 'support', 'Soporte (%)']].sort_values(
        ['longitud', 'support', 'Productos'], ascending=[True, False, True]
    ).reset_index(drop=True).rename(columns={'longitud': 'Longitud', 'support': 'Soporte'})

print('Conjuntos frecuentes con Apriori:', len(patrones_apriori))
display(tabla_patrones(patrones_apriori))


Conjuntos frecuentes con Apriori: 26


,Productos,Longitud,Canastas,Soporte,Soporte (%)
0,Pimientos,1,26,0.8667,86.6667
1,Zanahoria,1,26,0.8667,86.6667
2,Elotes,1,23,0.7667,76.6667
3,Pepino,1,22,0.7333,73.3333
4,Tomates,1,21,0.7000,70.0000
5,"Pimientos, Zanahoria",2,23,0.7667,76.6667
6,"Elotes, Pimientos",2,20,0.6667,66.6667
7,"Elotes, Zanahoria",2,20,0.6667,66.6667
8,"Pepino, Pimientos",2,19,0.6333,63.3333
9,"Pepino, Zanahoria",2,18,0.6000,60.0000


## 2.3. Patrones encontrados con FP-Growth

FP-Growth recibe exactamente la misma matriz y el mismo soporte minimo. Tambien se añade la longitud. El orden en que los conjuntos aparecen puede ser diferente al de Apriori; esto por si solo no indica que los resultados hayan cambiado.


In [21]:
patrones_fpgrowth = fpgrowth(matriz, min_support=MIN_SUPPORT, use_colnames=True)
patrones_fpgrowth['longitud'] = patrones_fpgrowth['itemsets'].apply(len)
print('Conjuntos frecuentes con FP-Growth:', len(patrones_fpgrowth))
display(tabla_patrones(patrones_fpgrowth))


Conjuntos frecuentes con FP-Growth: 26


,Productos,Longitud,Canastas,Soporte,Soporte (%)
0,Pimientos,1,26,0.8667,86.6667
1,Zanahoria,1,26,0.8667,86.6667
2,Elotes,1,23,0.7667,76.6667
3,Pepino,1,22,0.7333,73.3333
4,Tomates,1,21,0.7000,70.0000
5,"Pimientos, Zanahoria",2,23,0.7667,76.6667
6,"Elotes, Pimientos",2,20,0.6667,66.6667
7,"Elotes, Zanahoria",2,20,0.6667,66.6667
8,"Pepino, Pimientos",2,19,0.6333,63.3333
9,"Pepino, Zanahoria",2,18,0.6000,60.0000


## 2.4. Comparacion cuantitativa y verificacion de soportes

Ahora bien, se compara cada conjunto por sus productos y no por la posicion que ocupa en la tabla. Se revisan el numero de patrones, la cantidad por longitud y el soporte de cada uno. Ademas, los soportes se comprueban mediante conteos directos de las canastas originales.


In [22]:
a = patrones_apriori.assign(Conjunto=patrones_apriori['itemsets'].apply(lambda c: tuple(sorted(c))))
f = patrones_fpgrowth.assign(Conjunto=patrones_fpgrowth['itemsets'].apply(lambda c: tuple(sorted(c))))
comparacion = a[['Conjunto', 'support']].merge(
    f[['Conjunto', 'support']], on='Conjunto', how='outer',
    suffixes=('_Apriori', '_FP_Growth'), indicator=True, validate='one_to_one'
)
comparacion['Diferencia absoluta'] = (comparacion['support_Apriori'] - comparacion['support_FP_Growth']).abs()
coinciden_conjuntos = comparacion['_merge'].eq('both').all()
coinciden_soportes = np.allclose(comparacion['support_Apriori'], comparacion['support_FP_Growth'], atol=1e-12, rtol=0)
assert coinciden_conjuntos and coinciden_soportes

verificacion_directa = {}
for longitud in range(1, matriz.shape[1] + 1):
    for conjunto in combinations(matriz.columns, longitud):
        soporte = matriz[list(conjunto)].all(axis=1).mean()
        if soporte >= MIN_SUPPORT:
            verificacion_directa[frozenset(conjunto)] = soporte
for patrones in [patrones_apriori, patrones_fpgrowth]:
    assert set(patrones['itemsets']) == set(verificacion_directa)
    assert all(np.isclose(fila.support, verificacion_directa[fila.itemsets], atol=1e-12, rtol=0)
               for fila in patrones.itertuples())

por_longitud = pd.DataFrame({
    'Apriori': patrones_apriori.groupby('longitud').size(),
    'FP-Growth': patrones_fpgrowth.groupby('longitud').size()
}).reindex(range(1, matriz.shape[1] + 1), fill_value=0).fillna(0).astype(int)
por_longitud.index.name = 'Longitud'
print('Cantidad de patrones:', len(patrones_apriori), 'en cada algoritmo')
print('Coinciden los conjuntos:', bool(coinciden_conjuntos))
print('Coinciden los soportes:', bool(coinciden_soportes))
print('Diferencia maxima de soporte:', comparacion['Diferencia absoluta'].max())
print('Verificacion contra conteos directos: correcta')
display(por_longitud)
comparacion_visible = comparacion.drop(columns='_merge').copy()
comparacion_visible['Conjunto'] = comparacion_visible['Conjunto'].apply(lambda c: ', '.join(c))
display(comparacion_visible.sort_values('Conjunto').reset_index(drop=True))


Cantidad de patrones: 26 en cada algoritmo
Coinciden los conjuntos: True
Coinciden los soportes: True
Diferencia maxima de soporte: 0.0
Verificacion contra conteos directos: correcta


,Apriori,FP-Growth
Longitud,,
1,5,5
2,10,10
3,9,9
4,2,2
5,0,0


,Conjunto,support_Apriori,support_FP_Growth,Diferencia absoluta
0,Elotes,0.7667,0.7667,0.0
1,"Elotes, Pepino",0.5667,0.5667,0.0
2,"Elotes, Pepino, Pimientos",0.5000,0.5000,0.0
3,"Elotes, Pepino, Pimientos, Zanahoria",0.4333,0.4333,0.0
4,"Elotes, Pepino, Zanahoria",0.4667,0.4667,0.0
5,"Elotes, Pimientos",0.6667,0.6667,0.0
6,"Elotes, Pimientos, Tomates",0.4667,0.4667,0.0
7,"Elotes, Pimientos, Tomates, Zanahoria",0.4333,0.4333,0.0
8,"Elotes, Pimientos, Zanahoria",0.6000,0.6000,0.0
9,"Elotes, Tomates",0.5333,0.5333,0.0


Los dos algoritmos encuentran 26 conjuntos frecuentes. De estos, cinco tienen un producto, diez tienen dos, nueve tienen tres y dos tienen cuatro. No aparece un conjunto de cinco productos con el soporte minimo solicitado. Los conjuntos y sus soportes coinciden; la diferencia maxima es cero.

Como ejemplo del limite, Pepino, Tomates y Zanahoria aparecen juntos en 12 canastas. Su soporte es exactamente 0.40, por lo que este conjunto se conserva. La verificacion por conteo directo permite comprobar que los resultados vienen de las compras registradas.


## 2.5. Comparacion del tiempo de ejecucion

Tambien se mide el tiempo de extraccion para atender la parte computacional del objetivo. Primero se ejecuta cada algoritmo una vez para calentamiento. Despues se realizan 30 mediciones por algoritmo, alternando el orden. El tiempo no incluye cargar el Excel, instalar bibliotecas ni mostrar tablas.

Se reportan mediana y cuartiles porque una sola ejecucion puede variar por la carga del entorno. Son mediciones de esta muestra y de esta sesion de Colab; no permiten afirmar por si solas cual algoritmo sera mas rapido con un catalogo grande.


In [23]:
algoritmos = {'Apriori': apriori, 'FP-Growth': fpgrowth}
for funcion in algoritmos.values():
    funcion(matriz, min_support=MIN_SUPPORT, use_colnames=True)
mediciones = []
for repeticion in range(REPETICIONES):
    orden = list(algoritmos.items())
    if repeticion % 2:
        orden.reverse()
    for nombre, funcion in orden:
        inicio = perf_counter()
        funcion(matriz, min_support=MIN_SUPPORT, use_colnames=True)
        tiempo_ms = (perf_counter() - inicio) * 1000
        mediciones.append({'Algoritmo': nombre, 'Tiempo (ms)': tiempo_ms})
mediciones = pd.DataFrame(mediciones)
tiempos = mediciones.groupby('Algoritmo')['Tiempo (ms)'].agg(
    Repeticiones='size', Mediana_ms='median',
    Q1_ms=lambda x: x.quantile(0.25), Q3_ms=lambda x: x.quantile(0.75)
)
display(tiempos)
menor_mediana = tiempos['Mediana_ms'].idxmin()
print('Menor mediana en esta sesion:', menor_mediana)
print('Esta comparacion se limita a 30 canastas y cinco productos.')


,Repeticiones,Mediana_ms,Q1_ms,Q3_ms
Algoritmo,,,,
Apriori,30,6.0772,5.5697,8.7277
FP-Growth,30,10.4697,8.7133,12.9887


Menor mediana en esta sesion: Apriori
Esta comparacion se limita a 30 canastas y cinco productos.


## 2.6. Preguntas de la segunda seccion

### ¿Coinciden en cantidad y soporte los itemsets frecuentes extraidos por Apriori y FP-Growth? Justifica teoricamente el resultado.

Si. Ambos obtienen los mismos 26 conjuntos frecuentes y el mismo soporte para cada conjunto, utilizando la misma matriz y min_support = 0.40. La comparacion por nombre de productos y el conteo directo de las canastas lo confirman. El orden de las filas puede ser distinto, pero no cambia los patrones encontrados.

Esto ocurre porque los dos buscan todos los conjuntos que cumplen el mismo umbral de frecuencia. Cambia el procedimiento de busqueda, no la definicion de soporte. Apriori puede descartar candidatos a partir de sus subconjuntos: si un subconjunto no alcanza el soporte minimo, ningun conjunto mayor que lo contenga puede alcanzarlo, porque no puede aparecer en mas canastas. FP-Growth obtiene los patrones a partir de una representacion comprimida y sus arboles condicionales. Con la misma entrada, el mismo umbral y sin limitar la longitud, deben obtener el mismo conjunto de resultados.

### Explica la diferencia tecnica entre Apriori y FP-Growth. ¿Cual esperarias que escale mejor si el catalogo crece a miles de productos?

Apriori trabaja por niveles. Primero revisa conjuntos de un producto, despues de dos y asi sucesivamente. Forma candidatos y elimina los que contienen subconjuntos no frecuentes antes de calcular sus soportes. Cuando hay muchas combinaciones posibles, generar candidatos y contar sus apariciones puede aumentar el costo.

FP-Growth comprime las transacciones en un FP-Tree, donde las compras que comparten prefijos pueden compartir parte del arbol. Posteriormente obtiene patrones mediante bases y arboles condicionales, de forma recursiva, sin generar candidatos como Apriori.

Para un catalogo grande, esperariamos que FP-Growth escale mejor cuando haya muchos candidatos o prefijos compartidos. Esto no es una garantia para cualquier base: influyen el soporte minimo, la longitud y distribucion de las canastas, la cantidad de patrones frecuentes, la implementacion y la memoria disponible. Si las canastas comparten poco, la compresion puede ser menor. Los tiempos obtenidos con solo 30 clientes y cinco productos describen esta muestra; no demuestran el comportamiento con miles de productos.


## 2.7. Resultados guardados para continuar

Se conservan la matriz, las frecuencias y las salidas de ambos algoritmos. Estos resultados serviran despues para generar las reglas de asociacion. En este avance no se calculan reglas ni se seleccionan promociones.


In [24]:
carpeta_resultados = Path('resultados_secciones_1_2')
carpeta_resultados.mkdir(exist_ok=True)
matriz.to_csv(carpeta_resultados / '01_matriz_booleana.csv', encoding='utf-8-sig')
inspeccion.to_csv(carpeta_resultados / '02_inspeccion.csv', encoding='utf-8-sig')
frecuencias.to_csv(carpeta_resultados / '03_frecuencias_productos.csv', index=False, encoding='utf-8-sig')
tabla_patrones(patrones_apriori).to_csv(carpeta_resultados / '04_patrones_apriori.csv', index=False, encoding='utf-8-sig')
tabla_patrones(patrones_fpgrowth).to_csv(carpeta_resultados / '05_patrones_fpgrowth.csv', index=False, encoding='utf-8-sig')
comparacion_visible.to_csv(carpeta_resultados / '06_comparacion.csv', index=False, encoding='utf-8-sig')
por_longitud.to_csv(carpeta_resultados / '07_patrones_por_longitud.csv', encoding='utf-8-sig')
tiempos.to_csv(carpeta_resultados / '08_tiempos.csv', encoding='utf-8-sig')
print('Secciones 1 y 2 ejecutadas y verificadas.')
print('Resultados guardados en:', str(carpeta_resultados))


Secciones 1 y 2 ejecutadas y verificadas.
Resultados guardados en: resultados_secciones_1_2


## Fuentes de las dos secciones

- Carrasco Zuñiga, G. (2026). Practica 3: Machine Learning II. Guia proporcionada en Moodle, secciones 1 y 2.
- Archivo de datos base proporcionado con la guia: hoja Hoja1, 30 registros de clientes.
- [Documentacion de Apriori y codificacion transaccional, mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/).
- [Documentacion de FP-Growth, mlxtend](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/fpgrowth/).
